### Import

In [3]:
import re
import os
import gc
import html
import json
import string
import inflect
import warnings
import unicodedata
import contractions

import numpy as np
import pandas as pd
import datetime as dt
from tqdm.auto import tqdm

import seaborn as sns
import matplotlib.pyplot as plot

from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import DataLoader

import evaluate
import datasets

from datasets import Dataset
from datasets import DatasetDict
from datasets import load_dataset
from datasets import list_metrics

from transformers import AdamW
from transformers import get_scheduler

from transformers import AutoTokenizer
from transformers import DataCollatorWithPadding
from transformers import AutoModelForSequenceClassification

pd.options.display.max_rows = 1000
pd.options.display.max_colwidth = 10000
pd.set_option('display.max_columns', 500)

### General Path

In [4]:
mimiciv  = "PATH TO DATA/mimiciv(2.2)/note/"
abbreviation_path = '../Data/Abbreviations/Abbreviation.txt'
path_data= "../Data/EHR/"
path_out = '../Data/TEXT/'

### Read Data

In [5]:
all_note_ids = pd.read_csv(path_data + 'all_note_ids.csv', low_memory=False, index_col=False)
all_note_ids = all_note_ids.rename(columns={"radiology_note": "note_id"})
all_note_ids.head(2)

In [6]:
print(all_note_ids.stay_id.nunique())
print(all_note_ids.shape)

55604
(268328, 9)


### Prepare Note IDs in EHR Records

In [7]:
list_of_all_patients = all_note_ids.groupby('stay_id')[['stay_id', 'icu_expire_flag', 'hospital_expire_flag']].head(1)
list_of_all_patients = list_of_all_patients.reset_index(drop=True)
list_of_all_patients.head(2)

### Read Clinical Notes

In [9]:
reports = pd.read_csv(mimiciv + "radiology.csv")
reports = reports[['subject_id', 'hadm_id', 'storetime', 'charttime', 'note_id', 'text']]

In [10]:
reports['storetime'] = pd.to_datetime(reports['storetime'])
reports['charttime'] = pd.to_datetime(reports['charttime'])

condition_1 = (reports['charttime'].isnull())
reports.loc[condition_1, 'charttime'] = reports.loc[condition_1, 'storetime']

reports.drop(columns=['storetime'], inplace=True)

reports = reports.reset_index(drop=True)

### Final Reports

In [11]:
reports.head(1)

In [12]:
print("# table size = ", reports.shape)
print("# unique note = ", reports.note_id.nunique())
print("# unique patients = ", reports.subject_id.nunique())
print("# unique hospital admissions = ", reports.hadm_id.nunique())

# table size =  (2321355, 5)
# unique note =  2321355
# unique patients =  237427
# unique hospital admissions =  309670


### Filter Reports Based on ICU-Stay

In [13]:
all_reports = all_note_ids.merge(reports, on=['subject_id', 'hadm_id', 'note_id'], how='left')

In [14]:
all_reports['intime']  = pd.to_datetime(all_reports['intime'])
all_reports['outtime'] = pd.to_datetime(all_reports['outtime'])
all_reports['charttime'] = pd.to_datetime(all_reports['charttime'])

In [15]:
all_reports = all_reports[((all_reports['charttime'] >= all_reports['intime']) & (all_reports['charttime'] <= all_reports['outtime']))]

In [16]:
all_reports = all_reports[['subject_id', 'hadm_id', 'stay_id', 
                           'icu_expire_flag', 'hospital_expire_flag',
                           'note_id', 'text']]

all_reports['icu_expire_flag'] = all_reports['icu_expire_flag'].astype(int)
all_reports['hospital_expire_flag'] = all_reports['hospital_expire_flag'].astype(int)

all_reports = all_reports.drop_duplicates()

all_reports = all_reports.reset_index(drop=True)

In [17]:
all_reports.head(1)

In [18]:
print("# table size = ", all_reports.shape)
print("# unique note = ", all_reports.note_id.nunique())
print("# unique patients = ", all_reports.subject_id.nunique())
print("# unique hospital admissions = ", all_reports.hadm_id.nunique())
print("# unique icu admissions = ", all_reports.stay_id.nunique())

# table size =  (268214, 7)
# unique note =  268214
# unique patients =  41443
# unique hospital admissions =  50842
# unique icu admissions =  55585


### Convert Data to Dataset

In [19]:
df = Dataset.from_pandas(all_reports)

In [20]:
df

Dataset({
    features: ['subject_id', 'hadm_id', 'stay_id', 'icu_expire_flag', 'hospital_expire_flag', 'note_id', 'text'],
    num_rows: 268214
})

### Remove Patterns

In [21]:
def pattern_repl(matchobj):

    return ' '.rjust(len(matchobj.group(0)))

def find_end(text):
    
    ends = [len(text)]
    patterns = [
        re.compile(r'BY ELECTRONICALLY SIGNING THIS REPORT', re.I),
        re.compile(r'\n {3,}DR.', re.I),
        re.compile(r'[ ]{1,}RADLINE ', re.I),
        re.compile(r'.*electronically signed on', re.I),
        re.compile(r'M\[0KM\[0KM')]
    
    for pattern in patterns:
        matchobj = pattern.search(text)
        if matchobj:
            ends.append(matchobj.start())
    return min(ends)

def remove_pattern(text):

    text = re.sub(r'\[\*\*.*?\*\*\]', pattern_repl, text)
    text = re.sub(r'_', ' ', text)

    start = 0
    end = find_end(text)
    new_text = ''
    if start > 0:
        new_text += ' ' * start
    new_text = text[start:end]

    if len(text) - end > 0:
        new_text += ' ' * (len(text) - end)
    return new_text

### Filter based on Titles

In [22]:
SECTION_TITLES = re.compile(
                r'(ABDOMEN AND PELVIS|CLINICAL HISTORY|CLINICAL INDICATION|COMPARISON|COMPARISON STUDY DATE'
                r'|EXAM|EXAMINATION|FINDINGS|HISTORY|IMPRESSION|INDICATION|TRANSFER NOTE'
                r'|MEDICAL CONDITION|PROCEDURE|REASON FOR EXAM|REASON FOR STUDY|REASON FOR THIS EXAMINATION'
                r'|PORTABLE CHEST|ASSESSMENT|INTERPRETATION|CONCLUSIONS|REASON|ADMITTING DIAGNOSIS|STUDY|WET READ|NURSING ACCEPTANCE|ASSESS|PLAN'
                r'|TECHNIQUE'  
                r'|NEURO|CV|GI|GU|GI/GU|SKIN|IVF|RESP|ENOD|A/P'
                r'):|FINAL REPORT',
                re.I | re.M)

def has_meaningful_content(content):
    
    return bool(re.search(r'\w+', content))

def split_and_concatenate_title(text):
    
    sections = []
    matches = list(SECTION_TITLES.finditer(text))
    
    if not matches:
        return text
    else:
        for i, match in enumerate(matches):
            title = match.group().strip()
            content_start = match.end()
            content_end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
            content = text[content_start:content_end].strip()

            if has_meaningful_content(content):
                if sections and not text[matches[i - 1].end():match.start()].strip():
                    sections[-1] += "\n" + title + "\n" + content
                else:
                    sections.append(title + "\n" + content)
        return "\n".join(sections)

### Raw Text Length

In [23]:
def compute_text_length(text):
    
    len_text = len(text.split())
    
    return len_text

### Remove Extra Space 

In [24]:
def remove_wspaceA(text):
    
    clean_text = text.strip()
    clean_text = " ".join(clean_text.split())
    
    return clean_text

def remove_wspaceB(text):
    
    clean_text = re.sub('\s+', ' ', text)
    
    return clean_text

### Accented Characters to ASCII

In [25]:
def ascii_convert(text):
    
    clean_text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8', 'ignore')
    
    return clean_text

### Decontraction

In [26]:
def decontractionA(text):
    
    clean_text = contractions.fix(text)
    
    return clean_text

In [27]:
def decontractionB(phrase):

    phrase = re.sub(r"won\'t", "will not", phrase)
    phrase = re.sub(r"can\'t", "can not",  phrase)
    phrase = re.sub(r"won\’t", "will not", phrase)
    phrase = re.sub(r"can\’t", "can not",  phrase)
    phrase = re.sub(r"n\'t", " not",  phrase)
    phrase = re.sub(r"\'re", " are",  phrase)
    phrase = re.sub(r"\'s" , " is",   phrase)
    phrase = re.sub(r"\'d" , " would",phrase)
    phrase = re.sub(r"\'ll", " will", phrase)
    phrase = re.sub(r"\'t" , " not",  phrase)
    phrase = re.sub(r"\'ve", " have", phrase)
    phrase = re.sub(r"\'m" , " am",   phrase)
    phrase = re.sub(r"n\’t", " not",  phrase)
    phrase = re.sub(r"\’re", " are",  phrase)
    phrase = re.sub(r"\’s" , " is",   phrase)
    phrase = re.sub(r"\’d" , " would",phrase)
    phrase = re.sub(r"\’ll", " will", phrase)
    phrase = re.sub(r"\’t" , " not",  phrase)
    phrase = re.sub(r"\’ve", " have", phrase)
    phrase = re.sub(r"\’m" , " am",   phrase)

    return phrase

### Clinical Abbreviation

In [28]:
with open(abbreviation_path, 'r') as file:
    lines = file.readlines()
    
abbreviations = {}

for line in lines:
    parts = line.strip().split(' ', 1)
    abbr, full_form = parts
    if len(abbr) >= 3:  
        abbreviations[abbr] = full_form.strip()

In [29]:
def expand_abbreviations(text):
    
    pattern = re.compile(r'\b(' + '|'.join(re.escape(key) for key in abbreviations.keys()) + r')\b')
    clean_text  = pattern.sub(lambda x: abbreviations[x.group()], text)
    
    return clean_text

### Remove Punctuation

In [30]:
def replace_punc_with_space(text):

    mypunctuation = '!"#%&\'*+,/;<=>?@\\^_`|~'
    
    translation_table = str.maketrans(mypunctuation, ' ' * len(mypunctuation))
    
    clean_text = text.translate(translation_table)
    
    return clean_text

### Sentence & Word Count

In [31]:
def get_word_count_text(text):
    
    sent_count = 0
    word_count = 0
    vocab = {}
    
    sentences = sent_tokenize(str(text).lower())
    sent_count = sent_count + len(sentences)
    
    for sentence in sentences:
        words = word_tokenize(sentence)
        for word in words:
            if(word in vocab.keys()):
                vocab[word] = vocab[word] +1
            else:
                vocab[word] =1 
    word_count = len(vocab.keys())
    
    return word_count

In [32]:
def get_sentence_count_text(text):
    
    sent_count = 0
    word_count = 0
    vocab = {}
    
    sentences = sent_tokenize(str(text).lower())
    sent_count = sent_count + len(sentences)
    
    for sentence in sentences:
        words = word_tokenize(sentence)
        for word in words:
            if(word in vocab.keys()):
                vocab[word] = vocab[word] +1
            else:
                vocab[word] =1 
    word_count = len(vocab.keys())
    
    return sent_count

### Apply Procesing Functions

In [ ]:
df = df.map(lambda x: {"CLEAN_TEXT": remove_pattern(x["text"])})
df = df.map(lambda x: {"CLEAN_TEXT": remove_wspaceA(x["CLEAN_TEXT"])})
df = df.map(lambda x: {"CLEAN_TEXT": remove_wspaceB(x["CLEAN_TEXT"])})
df = df.map(lambda x: {"CLEAN_TEXT": ascii_convert(x["CLEAN_TEXT"])})
df = df.map(lambda x: {"CLEAN_TEXT": [html.unescape(o) for o in x["CLEAN_TEXT"]]}, batched=True)
df = df.map(lambda x: {"CLEAN_TEXT": split_and_concatenate_title(x["CLEAN_TEXT"])})
df = df.map(lambda x: {"CLEAN_TEXT": decontractionA(x["CLEAN_TEXT"])})
df = df.map(lambda x: {"CLEAN_TEXT": decontractionB(x["CLEAN_TEXT"])})
df = df.map(lambda x: {"CLEAN_TEXT": expand_abbreviations(x["CLEAN_TEXT"])})
df = df.map(lambda x: {"CLEAN_TEXT": replace_punc_with_space(x["CLEAN_TEXT"])})
df = df.map(lambda x: {"CLEAN_TEXT": remove_wspaceA(x["CLEAN_TEXT"])})
df = df.map(lambda x: {"CLEAN_TEXT": remove_wspaceB(x["CLEAN_TEXT"])})
df = df.map(lambda x: {"n_words_raw": get_word_count_text(x["text"])})
df = df.map(lambda x: {"n_sents_raw": get_sentence_count_text(x["text"])})
df = df.map(lambda x: {"len_text_raw": compute_text_length(x["text"])})
df = df.map(lambda x: {"n_words_clean": get_word_count_text(x["CLEAN_TEXT"])})
df = df.map(lambda x: {"n_sents_clean": get_sentence_count_text(x["CLEAN_TEXT"])})
df = df.map(lambda x: {"len_text_clean": compute_text_length(x["CLEAN_TEXT"])})

### Check Clean Examples

In [34]:
df[0]

### Select Columns

In [35]:
selected_columns_raw   = ['stay_id', 'icu_expire_flag', 'hospital_expire_flag', 'note_id', 'text', 'len_text_raw']
selected_columns_clean = ['stay_id', 'icu_expire_flag', 'hospital_expire_flag', 'note_id', 'CLEAN_TEXT', 'len_text_clean']

raw_dataset   = df.remove_columns([col for col in df.column_names if col not in selected_columns_raw])
clean_dataset = df.remove_columns([col for col in df.column_names if col not in selected_columns_clean])

clean_dataset = clean_dataset.rename_column("CLEAN_TEXT", "text")

### Filter Short Text

In [36]:
raw_min_length = 15
clean_min_length = 10

raw_dataset = raw_dataset.filter(lambda x: x["len_text_raw"] > raw_min_length)
clean_dataset = clean_dataset.filter(lambda x: x["len_text_clean"] > clean_min_length)

raw_dataset = raw_dataset.remove_columns(['len_text_raw'])
clean_dataset = clean_dataset.remove_columns(['len_text_clean'])

Filter:   0%|          | 0/268214 [00:00<?, ? examples/s]

Filter:   0%|          | 0/268214 [00:00<?, ? examples/s]

In [37]:
raw_dataset

Dataset({
    features: ['stay_id', 'icu_expire_flag', 'hospital_expire_flag', 'note_id', 'text'],
    num_rows: 268128
})

In [38]:
clean_dataset

Dataset({
    features: ['stay_id', 'icu_expire_flag', 'hospital_expire_flag', 'note_id', 'text'],
    num_rows: 268063
})

### Add Indicator of having original Note

In [39]:
# raw_new_column   = [1] * len(raw_dataset)
# clean_new_column = [1] * len(clean_dataset)

# raw_dataset = raw_dataset.add_column("hasNotes", raw_new_column)
# clean_dataset = clean_dataset.add_column("hasNotes", clean_new_column)

### Add Missing Notes

In [40]:
def add_missing_notes(list_of_all_patients, dataset, target_column):
    
    tmp_df = dataset.to_pandas()
    raw_df = tmp_df.copy()
    
    new_rows = []
    
    
    for index, row in list_of_all_patients.iterrows():
        icustay_id = row['stay_id']
        
        if icustay_id not in raw_df['stay_id'].values:
            
            new_row = {'stay_id': icustay_id,
                       'icu_expire_flag': int(row['icu_expire_flag']),
                       'hospital_expire_flag': int(row['hospital_expire_flag']),
                       'hasNotes': int(0),
                       'note_id': str(np.random.randint(100000, 200000)), 
                       target_column : "No clinical text note is available for this patient's record."}
            
            new_rows.append(new_row)
    
    new_rows_df = pd.DataFrame(new_rows)
    updated_df = pd.concat([raw_df, new_rows_df], ignore_index=True)
    updated_dataset = Dataset.from_pandas(updated_df)
    
    return updated_dataset

In [41]:
# raw_dataset_all_icu   = add_missing_notes(list_of_all_patients, raw_dataset,   'text')
# clean_dataset_all_icu = add_missing_notes(list_of_all_patients, clean_dataset, 'text')

### Add Phrase

In [42]:
def add_phrase(text):
    
    phrase = "This is a clinical note from an ICU stay aimed at assessing the risk of mortality:\n"
    clean_text = phrase + text
    
    return clean_text

In [43]:
# clean_dataset_all_icu = clean_dataset_all_icu.map(lambda x: {"text": add_phrase(x["text"])})

### Save Datasets

In [44]:
raw_dataset.save_to_disk(path_out + "MIMICIV_RAW_NOTES")
clean_dataset.save_to_disk(path_out + "MIMICIV_CLEAN_NOTES")

# raw_dataset_all_icu.save_to_disk(path_out + "MIMICIV_RAW_NOTES_ALL_ICU_PROMT")
# clean_dataset_all_icu.save_to_disk(path_out + "MIMICIV_CLEAN_NOTES_ALL_ICU_PROMT")

Saving the dataset (0/1 shards):   0%|          | 0/268128 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/268063 [00:00<?, ? examples/s]